Recreating the steps for theoretical maximum usage values in the Navarro River watershed in Python.

Step 1 requires loading eWRIMS flat file and Navarro MDT, then determine how many unique days a water right can divert and/or store water. Divide the allotted face value by the number of days to calculate a superseason daily rate of diversion or storage.

In [1]:
import pandas as pd
import numpy as np
print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)
import datetime

Pandas version: 3.0.3
Numpy version: 2.5.1


In [2]:
ewrims_ff = pd.read_csv("../referenceFiles/ewrims_flat_file_pod.csv")
mdt = pd.read_csv("../referenceFiles/NV_2017_2024_MDT_2026-03-11.csv")



C:\Users\kcappotto\AppData\Local\Temp\16\ipykernel_2972\1436094816.py:1: DtypeWarning: Columns (0: POD_NUMBER, 1: OFF_STO_DIV_RATE_UNIT, 2: POD_NUMBER_GIS, 3: QUAD_MAP_NUMBER, 4: CERTIFICATE_ID, 5: PERMIT_ID, 6: LICENSE_ID, 7: PRIORITY_DATE, 8: REJECTION_DATE, 9: INCOMPLETE_STATEMENT, 10: SUB_TYPE, 11: INI_REPORTED_DIV_UNIT, 12: SUPPLEMENTAL_STATEMENT_CYCLE, 13: TYPE_OF_DIVERSION_FACILITY, 14: QOW_DIVERTED_UNIT, 15: MAX_RATE_OF_DIV_UNIT, 16: WATER_USE_MIN_UNIT, 17: WATER_USE_MAX_UNIT, 18: REQUEST_FOR_REVOCATION_RECD, 19: STATE_WELL_NUMBER, 20: SURFACE_WATER_DIVERSIONS, 21: EFFECTIVE_TO_DATE, 22: SEPERATELY_OWNED, 23: TYPE_OF_STOCK, 24: USE_DIRECT_DIV_RATE_UNITS, 25: POU_DEVELOPMENT_STATUS, 26: PERMIT_PERMIT_ID, 27: PERMIT_ORIGINAL_ISSUE_DATE, 28: COMPLETE_CONSTRUCTION_DATE, 29: COMPLETE_APPLIC_WATER_DATE, 30: LICENSE_LICENSE_ID, 31: LICENSE_ORIGINAL_ISSUE_DATE, 32: LICENSE_REQUEST_TYPE, 33: LICENSE_REQUESTED_DATE, 34: INSPECTION_DATE, 35: REPORT_DATE, 36: OFFER_SENT_DATE, 37: ACCEPTED_

In [3]:
#mean_div_cols = [col for col in mdt.columns if "MEAN_DIV" in col] # select columns in mdt that have "MEAN_DIV" and create a list (hence the square brackets)

# loop does the same as above, easier for me to understand
mean_div_cols = [] # create an empty list
for col in mdt.columns: # for all columns in mdt
    if "MEAN_DIV" in col: # look if "MEAN_DIV" appears in any of the column names
        mean_div_cols.append(col) # if true, append it to the created mean_div_cols list
type(mean_div_cols)

mdt_summary = (
    mdt.groupby("ORIGINAL_APPLICATION_NUMBER")[mean_div_cols] # group data by ORIGINAL_APPLICATION_NUMBER and subset the mdt to only the mean_div_cols, as defined above
    # the () looks down ORIGINAL_APPLICATION_NUMBER column and groups together based on those values (i.e., have the column of Application Numbers)
    # the [] attaches the list to each row created by grouping ORIGINAL_APPLICATION_NUMBER
    .sum() # sum ignores NaN (matches R argument na.rm = TRUE)
    .reset_index() # pandas grouping puts ORIGINAL_APPLICATION_NUMBER into index; reset_index puts it back into a df column
) 

mdt_summary = mdt_summary.rename(columns = {"ORIGINAL_APPLICATION_NUMBER" : "APPLICATION_NUMBER"}) # columns = {"OLD_NAME" : "NEW_NAME"}


#### Next: Join and Filter

In [4]:
# check that the field to join on exists in both data frames
# print("ewrims_ff columns:", ewrims_ff.columns.tolist())
# print("mdt_summary columns:", mdt_summary.columns.tolist()) 

NV_table = pd.merge(ewrims_ff, mdt_summary, on = "APPLICATION_NUMBER", how = "inner")

# Filter to those that have values in any of the specified fields
NV_table_filter = NV_table[
    NV_table["STORAGE_SEASON_START"].notna() |
    NV_table["STORAGE_SEASON_END"].notna() |
    NV_table["DIRECT_DIV_SEASON_START"].notna() |
    NV_table["DIRECT_DIV_SEASON_END"].notna()
]
# print(NV_table_filter.head())

# Then select only Active PODs
NV_table_filter = NV_table_filter[NV_table_filter["POD_STATUS"] == "Active"]
# print(NV_table_filter.head())

# Select necessary columns from data frame
# [[]] - inner brackets create list of column names, outer brackets select data from pandas data frame
mean_div_cols = [col for col in NV_table_filter.columns if "mean_div" in col.lower()] # select call columns in the table that have mean_div in the name, regardless of uppercase/lowercase

columns_to_pick = ["POD_ID", "LATITUDE", "LONGITUDE", "APPLICATION_NUMBER", 
    "APPLICATION_PRIMARY_OWNER", "WATER_RIGHT_STATUS", "SOURCE_NAME", 
    "TRIB_DESC", "WATERSHED", "PARTY_ID", "STORAGE_SEASON_START", 
    "STORAGE_SEASON_END", "DIRECT_DIV_SEASON_START", "DIRECT_DIV_SEASON_END", 
    "FACE_VALUE_AMOUNT", "FACE_VALUE_UNITS", "INI_REPORTED_DIV_AMOUNT", 
    "INI_REPORTED_DIV_UNIT"]

NV_table_filter = NV_table_filter[mean_div_cols + columns_to_pick]

print(NV_table_filter.head())

   JAN_MEAN_DIV  FEB_MEAN_DIV  MAR_MEAN_DIV  APR_MEAN_DIV  MAY_MEAN_DIV  \
0      0.000000      0.000000      0.000000      0.000000      1.175763   
1      0.000000      0.000000      0.000000      0.000000      0.020459   
2      0.000000      0.000000      0.000000      0.000000      0.000000   
3      0.351986      0.383393      0.333145      0.203321      0.288411   
4      0.000000      0.000000      0.000000      0.000000      0.988384   

   JUN_MEAN_DIV  JUL_MEAN_DIV  AUG_MEAN_DIV  SEP_MEAN_DIV  OCT_MEAN_DIV  ...  \
0      2.020275      1.551012      0.625375      0.000250      0.000000  ...   
1      0.020459      0.020459      0.020459      0.020459      0.024551  ...   
2      0.000000      0.000000      0.000000      0.000000      0.000000  ...   
3      0.284195      0.335327      0.268033      0.164145      0.191752  ...   
4      0.923968      0.970508      0.399147      0.088203      0.000000  ...   

         WATERSHED  PARTY_ID  STORAGE_SEASON_START  STORAGE_SEASON_E

#### Prorate value for diversion/storage season

[Information about datetime formatting](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-behavior). `%m` = month as zero-padded decimal number (01, 02, ...); `%d` = day of month as zero-padded decimal number (01, 02, ..., 31)

In [5]:
# Create df with water rights that have specified diversion and/or storage seasons
NV_table_filter_DivSto = NV_table_filter[
    NV_table_filter["DIRECT_DIV_SEASON_START"].notna() |
    NV_table_filter["STORAGE_SEASON_START"].notna()
]

# number of days in diversion season
# pd.to_datetime makes timestamp that includes HH:MM:SS (default 00:00:00). W/ .dt.date at end, time is stripped, leaving YYYY-MM-DD
NV_table_filter_DivSto["DIV_START"] = pd.to_datetime(NV_table_filter_DivSto["DIRECT_DIV_SEASON_START"], format = "%m/%d").dt.date # create new column, convert contents of DIRECT_DIV_SEASON_START to pandas datetime object
NV_table_filter_DivSto["DIV_END"] = pd.to_datetime(NV_table_filter_DivSto["DIRECT_DIV_SEASON_END"], format = "%m/%d").dt.date

# format date columns to be in datetime to eventually get day of year
start_yday = pd.to_datetime(NV_table_filter_DivSto["DIV_START"]).dt.dayofyear
end_yday = pd.to_datetime(NV_table_filter_DivSto["DIV_END"]).dt.dayofyear

# function to process each row's start/end days
def calculate_diversion_days(row):
    start = row["start_yday"]
    end = row["end_yday"]

    # take care of NAs (same purpose as is.na() from R)
    if pd.isna(start) or pd.isna(end):
        return None, 0
    
    # take care of cases where diversion season extends into new calendar year (e.g., Dec 1 through Feb 28)
    if start > end:
        # list(range(start, 367)) covers start to 366
        # list(range(1, end + 1)) covers 1 to end
        days_list = list(range(int(start), 367)) + list(range(1, int(end) + 1))
    else:
        # standard range (i.e., end date is greater than start date)
        days_list = list(range(int(start), int(end) + 1))

    return days_list, len(days_list) # for each row, function returns: list of days & length of the list

# map start_yday, end_yday and apply function; create df for easier row access for the function
temp_df = pd.DataFrame({"start_yday":start_yday, "end_yday":end_yday})

results = temp_df.apply(calculate_diversion_days, axis = 1) # function applied across each row b/c axis = 1 (for 2D array, operate on rows with axis = 1)
# function has two outputs for each row, pandas puts them into a tuple for each row

# imagine results looking like:
# results = [
#    ([305, 306, ..., 15], 77), (days_list, length)
#    ([120, 121, ..., 130], 11), (days_list, length)
#    (None, 0) (days_list, length)
# ]

# put results into original df
NV_table_filter_DivSto["DIV_DAYS_LIST"] = [r[0] for r in results] # for the zeroth element in each row of results, put that into the DIV_DAYS_LIST column for the corresponding row
NV_table_filter_DivSto["DIV_DAYS_LENGTH"] = [r[1] for r in results] # for the first element in each row of results, put that into the DIV_DAYS_LENGTH column for the corresponding row
# print(NV_table_filter_DivSto["DIV_DAYS_LENGTH"].head())

# Number of days in storage season
NV_table_filter_DivSto["STO_START"] = pd.to_datetime(NV_table_filter_DivSto["STORAGE_SEASON_START"], format = "%m/%d").dt.date # create new column, convert contents of STORAGE_SEASON_START to pandas datetime object
NV_table_filter_DivSto["STO_END"] = pd.to_datetime(NV_table_filter_DivSto["STORAGE_SEASON_END"], format = "%m/%d").dt.date

# format date columns to be in datetime to eventually get day of year
start_yday = pd.to_datetime(NV_table_filter_DivSto["STO_START"]).dt.dayofyear
end_yday = pd.to_datetime(NV_table_filter_DivSto["STO_END"]).dt.dayofyear

# function to process each row's start/end days
def calculate_storage_days(row):
    start = row["start_yday"]
    end = row["end_yday"]

    # take care of NAs (same purpose as is.na() from R)
    if pd.isna(start) or pd.isna(end):
        return None, 0
    
    # take care of cases where storage season extends into new calendar year (e.g., Dec 1 through Feb 28)
    if start > end:
        # list(range(start, 367)) covers start to 366
        # list(range(1, end + 1)) covers 1 to end
        days_list = list(range(int(start), 367)) + list(range(1, int(end) + 1))
    else:
        # standard range (i.e., end date is greater than start date)
        days_list = list(range(int(start), int(end) + 1))

    return days_list, len(days_list) # for each row, function returns: list of days & length of the list

# map start_yday, end_yday and apply function; create df for easier row access for the function
temp_df = pd.DataFrame({"start_yday":start_yday, "end_yday":end_yday})

results = temp_df.apply(calculate_storage_days, axis = 1)

# put results into original df
NV_table_filter_DivSto["STO_DAYS_LIST"] = [r[0] for r in results] # for the zeroth element in each row of results, put that into the STO_DAYS_LIST column for the corresponding row
NV_table_filter_DivSto["STO_DAYS_LENGTH"] = [r[1] for r in results] # for the first element in each row of results, put that into the STO_DAYS_LENGTH column for the corresponding row
# print(NV_table_filter_DivSto["STO_DAYS_LENGTH"].head())


C:\Users\kcappotto\AppData\Local\Temp\16\ipykernel_2972\1363415852.py:9: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  NV_table_filter_DivSto["DIV_START"] = pd.to_datetime(NV_table_filter_DivSto["DIRECT_DIV_SEASON_START"], format = "%m/%d").dt.date # create new column, convert contents of DIRECT_DIV_SEASON_START to pandas datetime object


#### Conditions to calculate length of superseason

R operations being rewritten in python

```
NV_table_filter_DivSto <- NV_table_filter_DivSto |> add_column(SUPERSEASON_LENGTH = 0)

for (i in 1:nrow(NV_table_filter_DivSto)) {
  
  
  if (NV_table_filter_DivSto$DIV_DAYS_LENGTH[i] == 0) {
    
    superseasonLength <- NV_table_filter_DivSto$STO_DAYS_LENGTH[i]
  } else if (NV_table_filter_DivSto$STO_DAYS_LENGTH[i] == 0) {
    
    superseasonLength <- NV_table_filter_DivSto$DIV_DAYS_LENGTH[i]
    
  } else {
    superseasonLength <- c(NV_table_filter_DivSto$DIV_DAYS_LIST[[i]],
                           NV_table_filter_DivSto$STO_DAYS_LIST[[i]]) |>
      unique() |>
      length() 
  }
  
  
  NV_table_filter_DivSto$SUPERSEASON_LENGTH[i] <- superseasonLength
}
```

In [11]:
NV_table_filter_DivSto["SUPERSEASON_LENGTH"] = 0 # add new column SUPERSEASON_LENGTH to end of df; each row has 0 as placeholder value
# print(NV_table_filter_DivSto["SUPERSEASON_LENGTH"].head())

# Recreate the steps of the for loop

#Check for NaNs and set them to empty lists if they exist (this will allow lists to concatenate when evaluated in else_statement)
div_lists = NV_table_filter_DivSto["DIV_DAYS_LIST"].apply(
    lambda x: x if isinstance(x, list) else [] # check each row. if row contains list, leave unchanged. if NaN, replace with empty list []
)
sto_lists = NV_table_filter_DivSto["STO_DAYS_LIST"].apply(
    lambda x: x if isinstance(x, list) else []
)

else_statement = (
    (div_lists + sto_lists) # the + operator concatenates the lists
    #(NV_table_filter_DivSto["DIV_DAYS_LIST"] + NV_table_filter_DivSto["STO_DAYS_LIST"]) 
    .apply(set) # apply() applies the function inside; set creates unordered collection of unique values (filtering out duplicate values) --> apply(set) equivalent to unique() in R
    .apply(len) # len returns number of items (i.e., counts the unique values)
)

# == compares values - in each row, look in the specified columns to see if the value is equal to 0
# wrapped in square brackets to create a list of True or False for each row, whether or not its value is 0 (conditions becomes a list [True, True, False, ..., True])
conditions = [
    NV_table_filter_DivSto["DIV_DAYS_LENGTH"] == 0,
    NV_table_filter_DivSto["STO_DAYS_LENGTH"] == 0,
]

# wrapped in square brackets to create a list of column names
choices = [
    NV_table_filter_DivSto["STO_DAYS_LENGTH"], # if DIV is 0
    NV_table_filter_DivSto["DIV_DAYS_LENGTH"], # if STO is 0
]

NV_table_filter_DivSto["SUPERSEASON_LENGTH"] = np.select(
    conditions, choices, default = else_statement
)
# np.select is alternative to if / else if / else in R loop. Evaluates the arguments in the order they appear in (). If conditions is true, takes that value; if not it evaluates choices;
# if that is not true, it takes the provided default.
# np.select() maps conditions with choices based on the index. For each row, check condition 1 (is DIV_DAYS_LENGTH equal to 0?). If True, grab STO_DAYS_LENGTH for that row. Check condition 2 (is
# STO_DAYS_LENGTH equal to 0?). If True, grab DIV_DAYS_LENGTH for that row. If neither condition is True, evaluate the else_statement.
# "If DIV is 0, give me the STO value. If STO is 0, give me the DIV value." If neither are 0, use the else_statement to determine the value.
# It is looking in columns to determine the answer for the different scenarios.



#### Calculate the Prorated Value

R operations being rewritten in python

```
NV_table_filter_DivSto <- NV_table_filter_DivSto |> add_column(PRORATED_VALUE = 0)

for (i in 1:nrow(NV_table_filter_DivSto)) {
  proratedValue <- NV_table_filter_DivSto$FACE_VALUE_AMOUNT[i] / NV_table_filter_DivSto$SUPERSEASON_LENGTH[i]
  
  NV_table_filter_DivSto$PRORATED_VALUE[i] <- proratedValue
}

NV_table_filter_DivSto <- NV_table_filter_DivSto |> add_column(PRORATED_VALUE_UNITS = "AF per SS DAY")

# check if any prorated values are NA
for (i in 1:nrow(NV_table_filter_DivSto)) {
  
  prorate_NA <- list(is.na(NV_table_filter_DivSto$PRORATED_VALUE[i]))
  
}
```

Best practice with pandas is to minimize use of loops and perform vectorized operations. The operations perform for all rows without a loop.

In [ ]:
NV_table_filter_DivSto["PRORATED_VALUE"] = 0 # create new column at end of df, assign value of 0 to all rows

# calculate prorated value
NV_table_filter_DivSto["PRORATED_VALUE"] = (
    NV_table_filter_DivSto["FACE_VALUE_AMOUNT"] / NV_table_filter_DivSto["SUPERSEASON_LENGTH"]
)

NV_table_filter_DivSto["PRORATED_VALUE_UNITS"] = "AF per SS DAY"

# check for NA in prorated values
has_nas = NV_table_filter_DivSto["PRORATED_VALUE"].isna().any() # look for NA in each row of PRORATED_VALUE and evaluate to True (if NA) or False. .any() looks at the result of .isna()
# and returns True if there are one or more NA, otherwise False.
print(f"Are there any NA values? {has_nas}")
#print(NV_table_filter_DivSto["PRORATED_VALUE"].head())

Are there any NA values? False


In [14]:
NV_table_filter_DivSto.to_csv("NV_proratedFaceValues_python.csv")